# SaaS Business Analytics — Data Quality & Cleaning

**Objective:** prepare reliable business datasets for KPI analysis, SQL modeling, and Power BI.

**Portfolio note:** This notebook is a rebuilt analysis layer based on an open-source SaaS analytics reference. Original attribution is preserved in `docs/ATTRIBUTION.md`. The analysis structure, validation checks, KPI definitions, and outputs are being customized for a Data Analyst portfolio.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path('..')
RAW = ROOT / 'data' / 'raw'
CLEAN = ROOT / 'data' / 'cleaned'
pd.set_option('display.max_columns', None)

In [ ]:
files = {
 'customers': 'dim_customers.csv', 'plans':'dim_plans.csv', 'subscriptions':'fact_subscriptions.csv',
 'payments':'fact_payments.csv', 'usage':'fact_usage.csv', 'support':'fact_support.csv'
}
raw = {k: pd.read_csv(RAW/f) for k,f in files.items()}
{k:v.shape for k,v in raw.items()}

## 1. Data quality profile
We check row counts, duplicate business keys, missing values, and referential integrity before using the data for business decisions.

In [ ]:
def profile(df, key=None):
    out = pd.DataFrame({'dtype': df.dtypes.astype(str), 'missing': df.isna().sum(), 'missing_pct': (df.isna().mean()*100).round(2)})
    if key: out.loc[key,'duplicate_key_rows'] = df[key].duplicated().sum()
    return out

profile(raw['customers'],'customer_id')

In [ ]:
integrity = {
 'subscription_customer_orphans': (~raw['subscriptions'].customer_id.isin(raw['customers'].customer_id)).sum(),
 'subscription_plan_orphans': (~raw['subscriptions'].plan_id.isin(raw['plans'].plan_id)).sum(),
 'payment_subscription_orphans': (~raw['payments'].subscription_id.isin(raw['subscriptions'].subscription_id)).sum(),
 'usage_customer_orphans': (~raw['usage'].customer_id.isin(raw['customers'].customer_id)).sum(),
 'support_customer_orphans': (~raw['support'].customer_id.isin(raw['customers'].customer_id)).sum(),
}
pd.Series(integrity, name='orphan_rows')

## 2. Business-safe cleaning rules
- Normalize categorical labels and IDs.
- Parse dates with coercion and flag invalid records.
- Keep refunded transactions as negative amounts so net cash impact remains visible.
- Remove invalid subscription records only when the business key relationship is broken or duration is negative.
- Preserve the cleaned CSVs as reproducible inputs for SQL/Power BI.

In [ ]:
# Load the supplied cleaned layer and standardize dates for analysis
customers = pd.read_csv(CLEAN/'customers_clean.csv', parse_dates=['Signup_Date'])
plans = pd.read_csv(CLEAN/'plans_clean.csv')
subscriptions = pd.read_csv(CLEAN/'subscriptions_clean.csv', parse_dates=['Start_Date','End_Date'])
payments = pd.read_csv(CLEAN/'payments_clean.csv', parse_dates=['Payment_Date'])
usage = pd.read_csv(CLEAN/'usage_clean.csv', parse_dates=['Event_Date'])
support = pd.read_csv(CLEAN/'support_clean.csv', parse_dates=['Ticket_Date'])

for df in [customers,plans,subscriptions,payments,usage,support]:
    df.columns = [c.strip() for c in df.columns]

subscriptions['subscription_days'] = (subscriptions['End_Date'] - subscriptions['Start_Date']).dt.days
subscriptions.loc[subscriptions['subscription_days'].isna(),'subscription_days'] = 0
subscriptions['subscription_days'] = subscriptions['subscription_days'].astype(int)

In [ ]:
quality_summary = pd.DataFrame({
 'customers': [len(customers), customers.Customer_Id.nunique(), customers.Customer_Id.isna().sum()],
 'subscriptions': [len(subscriptions), subscriptions.Subscription_Id.nunique(), subscriptions.Customer_Id.isna().sum()],
 'payments': [len(payments), payments.Payment_Id.nunique(), payments.Subscription_Id.isna().sum()],
 'usage': [len(usage), usage.Usage_Id.nunique(), usage.Customer_Id.isna().sum()],
 'support': [len(support), support.Ticket_Id.nunique(), support.Customer_Id.isna().sum()]
}, index=['rows','unique_keys','missing_fk'])
quality_summary

## 3. KPI-ready data layer
The next notebook uses these definitions: **gross paid revenue** = sum of Paid payment amounts; **refunds** = absolute value of Refunded amounts; **net collected revenue** = Paid + Refunded amounts (failed payments are excluded); **churn rate** = churned subscriptions / all subscriptions; **ARPU** = net collected revenue / customers with positive net revenue. Definitions are intentionally explicit so the dashboard and SQL layer can be reconciled.

In [ ]:
paid = payments[payments.Payment_Status.eq('Paid')].copy()
refunded = payments[payments.Payment_Status.eq('Refunded')].copy()
net_revenue = payments.loc[payments.Payment_Status.isin(['Paid','Refunded']), 'Amount'].sum()
summary = pd.Series({
 'gross_paid_revenue': paid.Amount.sum(),
 'refund_value': refunded.Amount.abs().sum(),
 'net_collected_revenue': net_revenue,
 'active_subscriptions': subscriptions.Status.eq('Active').sum(),
 'churned_subscriptions': subscriptions.Status.eq('Churned').sum(),
 'churn_rate_pct': subscriptions.Status.eq('Churned').mean()*100
})
summary.round(2)